# ДЗ 4. Операции, axis и broadcasting на данных «Дала Маркет»
Срок: **до понедельника 16.11, 19:00**.
Сдача: папка `hw04` репозитория `dsa-homework` (`hw04.ipynb`): `git add hw04` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем. Небольшой цикл по городам или категориям при сборке таблицы допустим.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())
```

Ваши ответы и объяснения:

1. Ответ: [1. 2. 3.], [0 1 1]. Мы делим элементы нашего массива на обычное и целочисленное деление.
2. Выйдет: [4 4 4], так как создаётся массив 3 на 4, и у нас сложение идёт по строкам
3. Я думал выйдет: [[0], [2], [4]], но ответ оказался другим. Я не обратил внимание, что функция reshape() меняет размер не конечного ответа, а массив из [0 1 2]
4. Выйдет 6., так как функция sqrt (возведение в корень) выдаст нам 1. 2. 3, а их сумма даст нам 6.
5. Выйдет 7.5. Все элементы массива возводится во вторую степень, и вычисляется среднее значение. 

In [1]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [2]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
month  = np.array([r["order_date"][:7] for r in rows])      # "2026-09" или "2026-10"
cat    = np.array([r["category"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])

print(amount.shape, np.unique(month))

(5463,) ['2026-09' '2026-10']


In [169]:
# Чистые данные: без дублей (оставляем первое вхождение order_id), без TEST, только оплаченные
_, first = np.unique(oid, return_index=True)
keep = np.zeros(len(oid), dtype=bool)
keep[first] = True
clean = keep & (cust != "TEST") & paid
print("строк всего:", len(oid), "| в чистых данных:", clean.sum())

строк всего: 5463 | в чистых данных: 4739


## [База] 1. Выручка по чистым данным
Сколько строк осталось в `clean`? Выручка за сентябрь и октябрь и изменение в %. Совпадает ли с ДЗ 2?

In [24]:
# 1
september = amount[clean & (month == "2026-09")]
october = amount[clean & (month == "2026-10")]
print(f"Осталось строк в clean = {clean.sum()}")
print(f"Выручка за сентябрь: {september.sum()} тенге  |  Выручка за октябрь: {october.sum()} тенге")
print(f"Изменение выручки: {(october.sum() - september.sum()) / september.sum() * 100:.2f} %")

Осталось строк в clean = 4739
Выручка за сентябрь: 35243410.0 тенге  |  Выручка за октябрь: 34419520.0 тенге
Изменение выручки: -2.34 %


Если брать результат выручку из категории челендж, то ответ с дз2 совпадает. Однако, если брать из категории стандарт, то они не совпадут, так как в стандарте у нас были не очищенные данные) 

## [База] 2. Таблица «категория × месяц»
Соберите таблицу 5 × 2 по образцу таблицы городов с урока (категории: `np.unique(cat)`).
Какая категория приносит больше всего выручки за два месяца (`axis`)? Какая упала сильнее всех в %?

In [67]:
# 2
unique_cat = np.unique(cat)
unique_month = np.unique(month)
cat_and_month = np.array([[amount[clean & (cat == u) & (month == m)].sum() for m in unique_month] for u in unique_cat])
print(f"Самая прибальная категория: {unique_cat[cat_and_month.sum(axis=1).argmax()]} - {cat_and_month.sum(axis=1).max()} тенге")
losses = np.array([(m2 - m1) / m1 * 100 for m1, m2 in cat_and_month])
print(f"Сильнее всех упала выручка у {unique_cat[losses.argmin()]} на {losses.min():.2f} %")

Самая прибальная категория: Электроника - 34370430.0 тенге
Сильнее всех упала выручка у Одежда на -7.21 %


## [База] 3. Средний чек по категориям в октябре
Соберите вторую таблицу: **число** заказов по категориям и месяцам (сумма маски, а не сумма amount).
Средний чек = выручка / число заказов, одной операцией над двумя таблицами. У какой категории он самый высокий?

In [89]:
# 3
cat_and_month_in_quantity = np.array([[clean[clean & (month == m) & (cat == u)].sum() for m in unique_month] for u in unique_cat])
mean_cat_and_month = cat_and_month / cat_and_month_in_quantity
print(unique_cat[mean_cat_and_month.argmax(axis=0)])

['Электроника' 'Электроника']


In [82]:
mean_cat_and_month

array([[10499.43277311, 10768.46638655],
       [ 4020.        ,  3815.75250836],
       [ 6187.08530806,  6158.59770115],
       [13520.79646018, 13045.59815951],
       [32772.74952919, 33600.1980198 ]])

## [Стандарт] 4. Доля городов в выручке месяца
Используйте `table` городов с урока. Доля каждого города в выручке своего месяца: `table / table.sum(axis=0)`.
Проверьте, что каждый столбец в сумме даёт 1. Как изменилась доля Шымкента?

In [ ]:
# 4

In [129]:
cities = np.unique(city)
table = np.array([[amount[clean & (city == c) & (month == m)].sum() for m in unique_month] for c in cities])
revenue_share = table / table.sum(axis=0)
print(revenue_share.sum(axis=0))
print(f"Доля Шымкента изменилась на {revenue_share[5, 1] - revenue_share[5, 0]} %") 

[1. 1.]
Доля Шымкента изменилась на -0.011855825342961729 %


## [Стандарт] 5. Кто дал падение
Вклад каждого города в общее падение выручки: изменение города / общее изменение. Сколько процентов падения даёт Шымкент?
Напишите менеджеру одну фразу: где искать причину.

In [143]:
# 5
cities_changes = table[:, 1] - table[:, 0]
total_change = cities_changes.sum()
print(f"Вклад Шымкента: {(cities_changes / total_change * 100)[5]:.1f} %")
(cities_changes / total_change * 100)

Вклад Шымкента: 62.5 %


array([  4.43505808, -17.22924177,  22.29545206,  -5.56749081,
        33.60157303,  62.46464941])

Надо искать причину падения выручки в Шымкенте, так как на нём приходится 62.5% общего снижения выручки

## [Стандарт] 6. Исправьте код от AI
AI написал код «доля сентября и октября внутри каждого города». Код падает. Объясните, почему, и исправьте.
```python
share = table / table.sum(axis=1)
```

In [146]:
# 6
share = table / table.sum(axis=1, keepdims=True)
share

array([[0.50357019, 0.49642981],
       [0.49756469, 0.50243531],
       [0.50513538, 0.49486462],
       [0.48788975, 0.51211025],
       [0.51971739, 0.48028261],
       [0.52991168, 0.47008832]])

Код падает так как не проходит под условия broadcasting, у table размер 6 на 2 (6, 2), а у table.sum() размер 1 на 6 (6,). Для того, чтобы мы смогли вычислить долю сентября и октября внутри каждого города мы должны добавить  необязательный параметр keepdims=True внутри функции sum() чтобы поменять размер с (6,) на (6, 1).

## [Челлендж] 7. Стандартизация суммы заказа
Для чистых оплаченных заказов посчитайте z-оценку: `z = (x - x.mean()) / x.std()`.
Сколько заказов с |z| > 3 и какая это доля? Проверьте, что у `z` среднее ≈ 0 и std = 1.

In [164]:
# 7
x = amount[clean]
z = (x - x.mean()) / x.std()
share_of_emissions = np.abs(z) > 3
print(f"Заказов: {share_of_emissions.sum()} | Доля: {share_of_emissions.mean() * 100}")
print(f"z среднее = {z.mean():.0f} | std = {z.std()}")

Заказов: 87 | Доля: 1.8358303439544206
z среднее = 0 | std = 1.0


## [Челлендж] 8. Масштабирование признаков (заготовка для ML)
Соберите матрицу `X` из двух столбцов: `amount` и `days` для чистых заказов без пропусков в `days` (`np.column_stack`).
Приведите каждый столбец к диапазону [0, 1] одной строкой: `(X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))`.
Проверьте min и max каждого столбца. Объясните своими словами, зачем модели нужны признаки в одном масштабе.

In [181]:
# 8
clean_new = clean & ~np.isnan(days)
X = np.column_stack((amount[clean_new], days[clean_new]))
X_norm = (X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))
print(X_norm.min(axis=0), X_norm.max(axis=0))

[0. 0.] [1. 1.]


Если честно, я не знаю что я написал в данном задании и что каждая строчка делает и для чего, но хотел бы понять